# 02 · Energy Analysis ⭐

This is the core analysis notebook for the thesis. It provides a deep dive into CPU and
Memory energy consumption across 18 languages, grouped by execution paradigm (AOT, JIT,
Interpreted).

**Units:** All energy values are in **Joules (J)** (converted from raw µJ at load time).

**Key questions:**
- Which languages are most energy-efficient?
- Do paradigms (AOT vs JIT vs Interpreted) differ significantly in energy consumption?
- How does energy vary across benchmarks?

**Statistical methodology:**
Benchmark distributions are typically right-skewed and non-normal. We therefore use:
- **Median** as the primary central-tendency measure
- **Kruskal-Wallis** (non-parametric ANOVA) for paradigm comparisons
- **Mann-Whitney U** (pairwise) with **Bonferroni correction** for post-hoc tests
- **Rank-biserial correlation** as the effect size measure

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from itertools import combinations
from pathlib import Path
%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 150, 'figure.figsize': (12, 6)})

In [ ]:
# ── Column names as they appear in results_clean_runs.csv ────────────────────
# Units already converted by notebooks/01_data_cleaning.ipynb
COL_CPU_ENERGY = 'cpu_energy_rapl_msr_component-package_0-j'
COL_MEM_ENERGY = 'memory_energy_rapl_msr_component-dram_0-j'
COL_TIME       = 'phase_time_syscall_system-system-s'
COL_CPU_CARBON = 'cpu_carbon_rapl_msr_component-package_0-g'
COL_MEM_CARBON = 'memory_carbon_rapl_msr_component-dram_0-g'

ALPHA = 0.05

FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR = Path('outputs')
OUTPUTS_DIR.mkdir(exist_ok=True)

LANG_DISPLAY = {
    'c': 'C', 'cpp': 'C++', 'csharp': 'C#', 'fsharp': 'F#',
    'nodejs': 'JavaScript', 'dart': 'Dart', 'erlang': 'Erlang',
    'go': 'Go', 'haskell': 'Haskell', 'java': 'Java', 'lua': 'Lua',
    'ocaml': 'OCaml', 'perl': 'Perl', 'php': 'PHP',
    'python': 'Python', 'ruby': 'Ruby', 'rust': 'Rust', 'swift': 'Swift',
}

PARADIGM = {
    'C': 'AOT', 'C++': 'AOT', 'C#': 'AOT', 'Dart': 'AOT', 'Go': 'AOT',
    'Haskell': 'AOT', 'Java': 'AOT', 'OCaml': 'AOT', 'Rust': 'AOT', 'Swift': 'AOT',
    'Erlang': 'JIT', 'F#': 'JIT', 'JavaScript': 'JIT', 'PHP': 'JIT', 'Ruby': 'JIT',
    'Lua': 'Interpreted', 'Perl': 'Interpreted', 'Python': 'Interpreted',
}

PARADIGM_COLORS = {'AOT': '#2980b9', 'JIT': '#e67e22', 'Interpreted': '#27ae60'}
PARADIGM_ORDER  = ['AOT', 'JIT', 'Interpreted']

# Data pre-cleaned by notebooks/01_data_cleaning.ipynb:
#   - Outliers removed per (language × benchmark) group, IQR fence on CPU energy + time
#   - Units already converted (J, s, g, MB, W)
df = pd.read_csv('../../results/results_clean_runs.csv')
df['language'] = df['language'].replace(LANG_DISPLAY)
df['paradigm'] = df['language'].map(PARADIGM)

print(f"Shape: {df.shape}")
print(f"Languages ({df['language'].nunique()}): {sorted(df['language'].unique())}")
print(f"Benchmarks ({df['benchmark'].nunique()}): {sorted(df['benchmark'].unique())}")
print("Units: energy=J | time=s | carbon=g | disk/net=MB | power=W")
df.head(3)

## 1. CPU Energy by Language

Boxplots sorted by median CPU energy (J). Lower is better (more energy-efficient).
Each paradigm group is shown separately to highlight within-group spread.

In [ ]:
lang_order_cpu = (df.groupby('language')[COL_CPU_ENERGY]
                    .median()
                    .sort_values()
                    .index.tolist())

fig, ax = plt.subplots(figsize=(15, 6))
bp = ax.boxplot(
    [df[df['language'] == lang][COL_CPU_ENERGY].values for lang in lang_order_cpu],
    labels=lang_order_cpu, patch_artist=True, notch=False,
    medianprops=dict(color='black', linewidth=2),
    flierprops=dict(marker='x', markerfacecolor='red', markersize=5, alpha=0.6),
)
for patch, lang in zip(bp['boxes'], lang_order_cpu):
    patch.set_facecolor(PARADIGM_COLORS[PARADIGM[lang]])
    patch.set_alpha(0.75)

for i, lang in enumerate(lang_order_cpu):
    med = df[df['language'] == lang][COL_CPU_ENERGY].median()
    ax.text(i + 1, med, f'{med:.1f}', ha='center', va='bottom', fontsize=7, color='black')

legend_handles = [mpatches.Patch(color=PARADIGM_COLORS[p], label=p, alpha=0.75)
                  for p in PARADIGM_ORDER]
ax.legend(handles=legend_handles, title='Paradigm', loc='upper left')
ax.set_title('CPU Energy by Language (sorted by median)', fontsize=13)
ax.set_xlabel('Language')
ax.set_ylabel('CPU Energy (J)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cpu_energy_by_language.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=False)
for ax, paradigm in zip(axes, PARADIGM_ORDER):
    langs = [l for l in lang_order_cpu if PARADIGM[l] == paradigm]
    data  = [df[df['language'] == l][COL_CPU_ENERGY].values for l in langs]
    bp = ax.boxplot(data, labels=langs, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2),
                    flierprops=dict(marker='x', markerfacecolor='red', markersize=5, alpha=0.6))
    for patch in bp['boxes']:
        patch.set_facecolor(PARADIGM_COLORS[paradigm])
        patch.set_alpha(0.75)
    ax.set_title(f'{paradigm} Languages')
    ax.set_ylabel('CPU Energy (J)' if paradigm == 'AOT' else '')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

fig.suptitle('CPU Energy by Paradigm Group (J)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cpu_energy_per_paradigm.png', bbox_inches='tight')
plt.show()

## 2. Memory Energy by Language

Same structure as CPU energy. Memory energy (J) reflects DRAM power draw, which varies
less across paradigms but reveals GC pressure and allocation patterns.

In [ ]:
lang_order_mem = (df.groupby('language')[COL_MEM_ENERGY]
                    .median()
                    .sort_values()
                    .index.tolist())

fig, ax = plt.subplots(figsize=(15, 6))
bp = ax.boxplot(
    [df[df['language'] == lang][COL_MEM_ENERGY].values for lang in lang_order_mem],
    labels=lang_order_mem, patch_artist=True,
    medianprops=dict(color='black', linewidth=2),
    flierprops=dict(marker='x', markerfacecolor='red', markersize=5, alpha=0.6),
)
for patch, lang in zip(bp['boxes'], lang_order_mem):
    patch.set_facecolor(PARADIGM_COLORS[PARADIGM[lang]])
    patch.set_alpha(0.75)

for i, lang in enumerate(lang_order_mem):
    med = df[df['language'] == lang][COL_MEM_ENERGY].median()
    ax.text(i + 1, med, f'{med:.3f}', ha='center', va='bottom', fontsize=7)

legend_handles = [mpatches.Patch(color=PARADIGM_COLORS[p], label=p, alpha=0.75)
                  for p in PARADIGM_ORDER]
ax.legend(handles=legend_handles, title='Paradigm', loc='upper left')
ax.set_title('Memory Energy by Language (sorted by median)', fontsize=13)
ax.set_xlabel('Language')
ax.set_ylabel('Memory Energy (J)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'mem_energy_by_language.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=False)
for ax, paradigm in zip(axes, PARADIGM_ORDER):
    langs = [l for l in lang_order_mem if PARADIGM[l] == paradigm]
    data  = [df[df['language'] == l][COL_MEM_ENERGY].values for l in langs]
    bp = ax.boxplot(data, labels=langs, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2),
                    flierprops=dict(marker='x', markerfacecolor='red', markersize=5, alpha=0.6))
    for patch in bp['boxes']:
        patch.set_facecolor(PARADIGM_COLORS[paradigm])
        patch.set_alpha(0.75)
    ax.set_title(f'{paradigm} Languages')
    ax.set_ylabel('Memory Energy (J)' if paradigm == 'AOT' else '')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

fig.suptitle('Memory Energy by Paradigm Group (J)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'mem_energy_per_paradigm.png', bbox_inches='tight')
plt.show()

## 3. CPU + Memory Energy Combined

Two complementary views:
1. **Stacked bar chart** — total energy (CPU + Memory) per language in Joules, split by component
2. **Scatter plot** — CPU vs Memory energy (J), to identify languages where one dominates

In [ ]:
agg = df.groupby('language')[[COL_CPU_ENERGY, COL_MEM_ENERGY]].median()
agg = agg.sort_values(COL_CPU_ENERGY)
agg['paradigm'] = agg.index.map(PARADIGM)

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(agg))
ax.bar(x, agg[COL_CPU_ENERGY], label='CPU Energy (J)', color='#2980b9', alpha=0.85)
ax.bar(x, agg[COL_MEM_ENERGY], bottom=agg[COL_CPU_ENERGY],
       label='Memory Energy (J)', color='#e74c3c', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(agg.index, rotation=45, ha='right')
ax.set_title('Total Energy (CPU + Memory) by Language — median across all benchmarks', fontsize=12)
ax.set_ylabel('Energy (J)')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'total_energy_stacked.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for paradigm in PARADIGM_ORDER:
    langs = [l for l in agg.index if agg.loc[l, 'paradigm'] == paradigm]
    ax.scatter(
        agg.loc[langs, COL_CPU_ENERGY],
        agg.loc[langs, COL_MEM_ENERGY],
        color=PARADIGM_COLORS[paradigm], label=paradigm, s=80, zorder=3
    )
    for lang in langs:
        ax.annotate(lang,
                    (agg.loc[lang, COL_CPU_ENERGY], agg.loc[lang, COL_MEM_ENERGY]),
                    textcoords='offset points', xytext=(6, 4), fontsize=8)

ax.set_title('CPU Energy vs Memory Energy — median per language (J)', fontsize=12)
ax.set_xlabel('CPU Energy (J)')
ax.set_ylabel('Memory Energy (J)')
ax.legend(title='Paradigm')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cpu_vs_mem_scatter.png', bbox_inches='tight')
plt.show()

## 4. Paradigm Comparison

**Violin plots** show the full distribution shape per paradigm.
**Kruskal-Wallis** tests whether any paradigm differs significantly.
If significant, **pairwise Mann-Whitney U** tests with **Bonferroni correction** identify which pairs differ.
Effect size is reported as **rank-biserial correlation** r = 1 − 2U/(n₁·n₂).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, col, label in zip(axes,
                           [COL_CPU_ENERGY, COL_MEM_ENERGY],
                           ['CPU Energy (J)', 'Memory Energy (J)']):
    groups = [df[df['paradigm'] == p][col].values for p in PARADIGM_ORDER]
    parts  = ax.violinplot(groups, positions=range(len(PARADIGM_ORDER)), showmedians=True)
    for pc, p in zip(parts['bodies'], PARADIGM_ORDER):
        pc.set_facecolor(PARADIGM_COLORS[p])
        pc.set_alpha(0.7)
    ax.set_xticks(range(len(PARADIGM_ORDER)))
    ax.set_xticklabels(PARADIGM_ORDER)
    ax.set_title(f'{label} by Paradigm')
    ax.set_ylabel(label)

fig.suptitle('Energy Distribution by Execution Paradigm', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'energy_violin_paradigm.png', bbox_inches='tight')
plt.show()

In [ ]:
def rank_biserial(x, y):
    """Rank-biserial correlation as effect size for Mann-Whitney U."""
    u, _ = stats.mannwhitneyu(x, y, alternative='two-sided')
    return 1 - (2 * u) / (len(x) * len(y))

for col, label in [(COL_CPU_ENERGY, 'CPU Energy (J)'), (COL_MEM_ENERGY, 'Memory Energy (J)')]:
    groups   = {p: df[df['paradigm'] == p][col].values for p in PARADIGM_ORDER}
    kw_stat, kw_p = stats.kruskal(*groups.values())
    n_pairs = len(PARADIGM_ORDER) * (len(PARADIGM_ORDER) - 1) // 2

    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"  Kruskal-Wallis H={kw_stat:.3f}, p={kw_p:.4f} ", end='')
    print("(SIGNIFICANT)" if kw_p < ALPHA else "(not significant)")

    if kw_p < ALPHA:
        print(f"  Post-hoc Mann-Whitney U (Bonferroni α={ALPHA/n_pairs:.4f}):")
        for (p1, p2) in combinations(PARADIGM_ORDER, 2):
            u, p = stats.mannwhitneyu(groups[p1], groups[p2], alternative='two-sided')
            p_adj = min(p * n_pairs, 1.0)
            r = rank_biserial(groups[p1], groups[p2])
            sig = "✓" if p_adj < ALPHA else "✗"
            print(f"    {sig} {p1} vs {p2}: U={u:.0f}, p_adj={p_adj:.4f}, r={r:.3f}")

## 5. Per-Benchmark Energy Heatmap

Heatmap of median CPU and Memory energy (J) for each language × benchmark combination.
This reveals which benchmarks are most energy-intensive and which languages suffer
disproportionately on specific workloads.

In [ ]:
pivot_cpu = df.groupby(['language', 'benchmark'])[COL_CPU_ENERGY].median().unstack()
pivot_mem = df.groupby(['language', 'benchmark'])[COL_MEM_ENERGY].median().unstack()

lang_sort = df.groupby('language')[COL_CPU_ENERGY].median().sort_values().index
pivot_cpu = pivot_cpu.loc[lang_sort]
pivot_mem = pivot_mem.loc[lang_sort]

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
for ax, pivot, title in zip(axes,
                              [pivot_cpu, pivot_mem],
                              ['CPU Energy (J) — median', 'Memory Energy (J) — median']):
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
                linewidths=0.3, cbar_kws={'label': 'Energy (J)'})
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Benchmark')
    ax.set_ylabel('Language')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

fig.suptitle('Per-Benchmark Energy Heatmap — sorted by median CPU energy (J)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'energy_heatmap_benchmark.png', bbox_inches='tight')
plt.show()

## 6. Energy Efficiency Ranking

Languages ranked by median CPU energy (J) ascending — lower rank = more efficient.
A combined rank averages CPU and Memory energy ranks.

In [ ]:
rank_agg = df.groupby('language').agg(
    paradigm       = ('paradigm', 'first'),
    cpu_mean_J     = (COL_CPU_ENERGY, 'mean'),
    cpu_median_J   = (COL_CPU_ENERGY, 'median'),
    mem_mean_J     = (COL_MEM_ENERGY, 'mean'),
    mem_median_J   = (COL_MEM_ENERGY, 'median'),
)
rank_agg['cpu_rank'] = rank_agg['cpu_median_J'].rank().astype(int)
rank_agg['mem_rank'] = rank_agg['mem_median_J'].rank().astype(int)
rank_agg['combined_rank'] = ((rank_agg['cpu_rank'] + rank_agg['mem_rank']) / 2).round(1)
ranking = rank_agg.sort_values('combined_rank')
ranking.index.name = 'Language'
ranking[['paradigm', 'cpu_median_J', 'mem_median_J', 'cpu_rank', 'mem_rank', 'combined_rank']]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
colors = [PARADIGM_COLORS[PARADIGM[l]] for l in ranking.index]
ax.barh(ranking.index, ranking['cpu_median_J'], color=colors, alpha=0.85, edgecolor='white')
ax.set_title('CPU Energy Efficiency Ranking — median across all benchmarks (J)', fontsize=12)
ax.set_xlabel('Median CPU Energy (J)')
ax.set_ylabel('Language')
ax.invert_yaxis()
legend_handles = [mpatches.Patch(color=PARADIGM_COLORS[p], label=p, alpha=0.85)
                  for p in PARADIGM_ORDER]
ax.legend(handles=legend_handles, title='Paradigm', loc='lower right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cpu_energy_ranking.png', bbox_inches='tight')
plt.show()

## 7. CO₂ Carbon Correlation

CPU carbon (g CO₂) is derived from CPU energy via a carbon-intensity factor.
We verify the correlation and check whether it is simply proportional or shows variance.
Values are in grams (g) — converted from raw µg at load time.

In [ ]:
cpu_carbon_nonzero = (df[COL_CPU_CARBON] != 0).mean()
print(f"Non-zero CPU carbon values: {cpu_carbon_nonzero:.1%}")

if cpu_carbon_nonzero > 0.5:
    spearman_r, spearman_p = stats.spearmanr(df[COL_CPU_ENERGY], df[COL_CPU_CARBON])
    print(f"Spearman r(CPU energy J, CPU carbon g) = {spearman_r:.4f}, p = {spearman_p:.2e}")

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.scatter(df[COL_CPU_ENERGY], df[COL_CPU_CARBON], alpha=0.3, s=20, color='steelblue')
    ax.set_xlabel('CPU Energy (J)')
    ax.set_ylabel('CPU Carbon (g CO₂)')
    ax.set_title(f'CPU Energy (J) vs CPU Carbon (g) — Spearman r={spearman_r:.3f}')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'cpu_energy_vs_carbon.png', bbox_inches='tight')
    plt.show()
else:
    print("CPU carbon column is mostly zero — likely not recorded for this dataset.")